# Diabetes Prediction using ML pipeline
Modified Notebook with Threshold optimization to handle false negative and false positives

## Load Libraries and Configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, f1_score
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


## Load and QC data

In [ ]:
# Loading data

df_train = pd.read_csv('/kaggle/input/playground-series-s5e12/train.csv')

df_test = pd.read_csv('/kaggle/input/playground-series-s5e12/test.csv')

TARGET = "diagnosed_diabetes"

X = train_df.drop(columns=[TARGET, "id"])
y = train_df[TARGET]

test_ids = test_df["id"]
X_test_final = test_df.drop(columns=["id"])


In [ ]:
# Feature Groups and Preprocessors
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(
            drop="first",
            handle_unknown="ignore",
            sparse_output=True   # <<< CRITICAL
        ), categorical_features)
    ],
    sparse_threshold=1.0
)

In [ ]:
# Logistic Regression

logreg_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logreg_pipe.fit(X_train, y_train)

logreg_prob = logreg_pipe.predict_proba(X_test)[:, 1]
logreg_auc = roc_auc_score(y_test, logreg_prob)

print(f"Logistic Regression ROC-AUC: {logreg_auc:.3f}")

In [ ]:
# Random Forest

rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=50,
        max_depth=12,
        class_weight="balanced",
        random_state=42
    ))
])

rf_pipe.fit(X_train, y_train)

rf_prob = rf_pipe.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_prob)

print(f"Random Forest ROC-AUC: {rf_auc:.3f}")

In [ ]:
# XGBoost Model

xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="logloss",
        random_state=42
    ))
])

xgb_pipe.fit(X_train, y_train)

xgb_prob = xgb_pipe.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_prob)

print(f"XGBoost ROC-AUC: {xgb_auc:.3f}")

In [ ]:
# LightGBM Model

lgbm_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.0125,
        num_leaves=71,
        subsample=0.8,
        colsample_bytree=0.8,
        force_col_wise=True,
        random_state=42
    ))
])

lgbm_pipe.fit(X_train, y_train)

lgbm_prob = lgbm_pipe.predict_proba(X_test)[:, 1]
lgbm_auc = roc_auc_score(y_test, lgbm_prob)

print(f"LightGBM ROC-AUC: {lgbm_auc:.3f}")

In [ ]:
# Collect Results
results_df = pd.DataFrame({
    "Model": ["Logistic", "RandomForest", "XGBoost", "LightGBM"],
    "ROC_AUC": [logreg_auc, rf_auc, xgb_auc, lgbm_auc]
}).sort_values("ROC_AUC", ascending=False)

results_df

In [ ]:
# Refit Best Model on full data
results_df = pd.DataFrame(
    results, columns=["Model", "Calibration", "ROC_AUC"]
).sort_values("ROC_AUC", ascending=False)

results_df

In [ ]:
# Validation Probabilities

val_prob = final_model.predict_proba(X_val)[:, 1]


In [ ]:
# Decision Threshold Optimization

thresholds = np.linspace(0.01, 0.99, 99)
records = []

for t in thresholds:
    y_pred = (val_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    f1 = f1_score(y_val, y_pred)

    records.append([t, sens, spec, f1])

thr_df = pd.DataFrame(
    records, columns=["Threshold", "Sensitivity", "Specificity", "F1"]
)

# Youden's

fpr, tpr, roc_thr = roc_curve(y_val, val_prob)
youden_thr = roc_thr[np.argmax(tpr - fpr)]


In [ ]:
# Cost-Sensitive thresholds

C_FN, C_FP = 5, 1
costs = []

for t in thresholds:
    y_pred = (val_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    costs.append(C_FN * fn + C_FP * fp)

cost_thr = thresholds[np.argmin(costs)]

In [ ]:
# Final Deployment

DEPLOYMENT_THRESHOLD = cost_thr
DEPLOYMENT_THRESHOLD


In [ ]:
# Deploy to Test Set

test_prob = final_model.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({
    "id": test_ids,
    "diabetes_probability": test_prob,
    "predicted_class": (test_prob >= DEPLOYMENT_THRESHOLD).astype(int)
})

submission.head()


In [ ]:
submission.to_csv("submission.csv", index=False)

print("Deployment file saved: submission.csv")